# Simple Baseline — leak-safe ElasticNet

**Stage 1 baseline — PMLDL 2026 project.**

## Attribution
This notebook is adapted from the public Kaggle notebook
<https://www.kaggle.com/code/morodertobias/hull-leak-safe-baseline>
for the *Hull Tactical Market Prediction* competition.
It also follows the Hull starter notebook for its feature list and model.

All modelling logic is the original author's. Changes made for this project are
limited to **data loading and results saving**, and every one of them is marked
with an `ADDED (PMLDL Stage 1)` comment:

1. hard-coded `/kaggle/input/...` paths replaced by the shared `src` loader;
2. training data trimmed to exclude the held-out public block (last 180
   `date_id`s), which the original notebooks trained on;
3. the `kaggle_evaluation` inference server replaced by an offline replay of
   that block, since the forecasting phase is out of scope;
4. scoring routed through `src.metrics` and persisted via `src.results`.

No author names, usernames or team identifiers are reproduced here.


In [ ]:
# ===================== ADDED (PMLDL Stage 1) =====================
# Shared project interface. Replaces all hard-coded /kaggle/input paths and
# ad-hoc scoring. Original modelling logic below is untouched.
# ==================================================================
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

from src import *          # see INTERFACE.md
import numpy as np, pandas as pd

set_seed()                 # SEED = 42

# --- universal loader: last 180 date_ids held out (public leaderboard block)
hull = load_dataset()
print(hull)

# --- ADDED: materialise a leak-safe train.csv / test.csv pair so the original
# code can keep using read_csv() without pointing at the public block.
PREPARED = RESULTS_DIR / "prepared"; PREPARED.mkdir(parents=True, exist_ok=True)
hull.train.to_csv(PREPARED / "train.csv", index=False)

_pub = hull.public.copy()
for _c in LOOKAHEAD_COLS:                      # rebuild the API's lagged_* columns
    _pub["lagged_" + _c] = _pub[_c].shift(1).fillna(0.0)
_pub["is_scored"] = True
_pub.drop(columns=list(LOOKAHEAD_COLS)).to_csv(PREPARED / "test.csv", index=False)

PUBLIC_FWD = hull.public["forward_returns"].to_numpy()
PUBLIC_RF  = hull.public["risk_free_rate"].to_numpy()
PUBLIC_Y   = hull.public[TARGET].to_numpy()

# ADDED (PMLDL): always include the same-metric passive index reference.
save_result(model="passive_w1", stage="benchmark", split="public",
            metrics=evaluate(PUBLIC_Y, np.zeros(len(PUBLIC_Y)),
                             weights=np.ones(len(PUBLIC_Y)),
                             forward_returns=PUBLIC_FWD, risk_free_rate=PUBLIC_RF),
            notes="Historically consulted public diagnostic; passive w=1")

def replay_public(predict_fn):
    """ADDED: offline stand-in for kaggle_evaluation - feeds the held-out
    public block to predict() one row at a time, exactly as the API would."""
    import polars as pl
    t = pl.read_csv(PREPARED / "test.csv")
    return np.array([float(predict_fn(t[i:i+1])) for i in range(t.height)])


# Hull - Leak Safe Baseline

- The training data also contain the public test set. It is the last 180 days, see [data description](https://www.kaggle.com/competitions/hull-tactical-market-prediction/data). Let's remove this part overall to get a meaningful score on the current public leaderboard.
- We are supposed to predict the strategy by day, which is something that depends on the forward_return and risk_free_rate (and some overall effect). This makes me wonder if we need to estimate both of them at the same time, and then deriving a certain stratgey. However, in [this discussion answer](https://www.kaggle.com/competitions/hull-tactical-market-prediction/discussion/608349#3299060), this optimization is done analytically (withouth considering penalty effects). Thus let's take these targets as the true targets to be predicted.
- Otherwise we are modelling as simple as in the [Hull Starter Notebook](https://www.kaggle.com/code/laurentlanteigne/hull-starter-notebook): No time effect, same features and same model.
- Note, the training dataset will be updated throughout the competition.

**All comments welcome!**

## Import & Settings

In [ ]:
import os
import pathlib
import numpy as np
import pandas as pd
import polars as pl 
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode, iplot
import plotly as py
init_notebook_mode(connected=True) 
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.preprocessing import StandardScaler
# ADDED: kaggle_evaluation / metric.py are competition-runtime only.
# Scoring is routed through the shared interface instead (see setup cell).
from src.metrics import modified_sharpe

def hull_score(solution, submission, row_id_column_name=""):
    return modified_sharpe(submission["prediction"].to_numpy(),
                           solution["forward_returns"].to_numpy(),
                           solution["risk_free_rate"].to_numpy())

In [ ]:
# ADDED: path + seed now come from src.config (SEED pinned to 42 project-wide).
BASE_DIR = DATA_DIR
TEST_SKIP = 180
# same features as in hull starter nb
FEATURES = [
    "S2",
    "E2", "E3",
    "P8", "P9", "P10", "P12", "P13",
    "S1", "S5", 
    "I2",
    "U1",
    "U2",
]
INFO_COLS = ["date_id", "forward_returns", "risk_free_rate"]
# model as in hull starter
CV = 10
L1_RATIO = 0.5
ALPHAS = np.logspace(-4, 2, 100)
MAX_ITER = 1000000

## Load data

In [ ]:
# ADDED: loaded via the shared loader instead of a raw read_csv.
data = hull.full.copy()
data["U1"] = data["I2"] - data["I1"]
data["U2"] = data["M11"] / ((data["I2"] + data["I9"] + data["I7"]) / 3)
data = data[FEATURES + INFO_COLS].dropna()
data

In [ ]:
max_train_date = data["date_id"].max() - TEST_SKIP
print("max train date_id:", max_train_date)

In [ ]:
train = data.loc[data["date_id"] <= max_train_date].copy()
test = data.loc[data["date_id"] > max_train_date].copy()
print("train shape:", train.shape)
print("test shape:", test.shape)

## Build target
Set target as best strategy on training dataset.

In [ ]:
solution = train.copy()
market_excess_returns = solution['forward_returns'] - solution['risk_free_rate']
market_excess_cumulative = (1 + market_excess_returns).prod()
market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solution)) - 1
c = (1 + market_mean_excess_return) ** (1 / (market_excess_returns > 0).mean()) - 1
submission = pd.DataFrame({'prediction': (c / market_excess_returns).clip(0, 2)})
print("best score train:", hull_score(solution, submission, ''))

In [ ]:
train["target"] = submission

In [ ]:
fig = go.Figure(data=[go.Scatter3d(
    x=train['forward_returns'],
    y=train['risk_free_rate'],
    z=train['target'],
    mode='markers',
    marker=dict(size=3)
)])
fig.update_layout(
    scene=dict(
        xaxis_title='forward_returns',
        yaxis_title='risk_free_rate',
        zaxis_title='target'
    )
)
iplot(fig)

In [ ]:
market_excess_returns = np.linspace(-0.01, 0.04, 402)
fig = go.Figure(data=[go.Scatter(
    x=market_excess_returns,
    y=(c / market_excess_returns).clip(0, 2),
    mode='markers',
    marker=dict(size=3)
)])
fig.update_layout(
    xaxis_title="market_excess_returns",
    yaxis_title="target",
)
iplot(fig)

## Model

In [ ]:
X_train = train[FEATURES].values
y_train = train["target"].values

In [ ]:
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
model_cv = ElasticNetCV(l1_ratio=L1_RATIO, cv=CV, alphas=ALPHAS, max_iter=MAX_ITER)
model_cv.fit(X_train_scaled, y_train)
model = ElasticNet(alpha=model_cv.alpha_, l1_ratio=L1_RATIO)
model.fit(X_train_scaled, y_train)

In [ ]:
model.score(X_train_scaled, y_train)

## Invoke on test set

In [ ]:
X_test = test[FEATURES].values
X_test_scaled = sc.transform(X_test)
y_test_pred = model.predict(X_test_scaled)
y_test_pred = np.clip(y_test_pred, 0.0, 2.0)
pd.Series(y_test_pred).describe()

In [ ]:
solution = test.copy()
submission = pd.DataFrame({'prediction': y_test_pred}, index=solution.index)
print("score public test:", hull_score(solution, submission, ''))

# Submit

In [ ]:
def predict(test: pl.DataFrame) -> float:
    data = test.to_pandas()
    data["U1"] = data["I2"] - data["I1"]
    data["U2"] = data["M11"] / ((data["I2"] + data["I9"] + data["I7"]) / 3)    
    X = data[FEATURES].values
    X_scaled = sc.transform(X)
    y = model.predict(X_scaled)
    pred = np.clip(y, 0.0, 2.0)[0]
    print(f"date_id: {data['date_id'][0]} -> prediction: {pred:>.4f}")
    return pred

In [ ]:
# ADDED: inference server removed - public phase only, replayed offline in the
# final cell via replay_public(predict).

In [ ]:
MODEL_NAME = "elasticnet_leak_safe"
MODEL_NOTES = "ElasticNet on analytic optimal-allocation target; public block held out"

# ===================== ADDED (PMLDL Stage 1) =====================
# Universal results saver. Replays the held-out public block through predict()
# and records the run in results/leaderboard.csv.
# ==================================================================
allocations = replay_public(predict)

metrics = evaluate(
    PUBLIC_Y,
    allocations,               # allocation doubles as the ranking signal here
    weights=allocations,
    forward_returns=PUBLIC_FWD,
    risk_free_rate=PUBLIC_RF,
)
print({k: round(v, 4) for k, v in metrics.items() if isinstance(v, float)})

_ = save_result(
    model=MODEL_NAME, stage="baseline", metrics=metrics,
    split="public", notes=MODEL_NOTES,
)
# ADDED (PMLDL): retain aligned allocations for comparison and bootstrap.
save_predictions(MODEL_NAME, hull.public[DATE_COL], PUBLIC_Y, allocations, allocations)


In [ ]:
display(compare(split="public"))